### Exploratory Data Analysis (EDA): UK Aviation Recovery (2021-2024)


**Data Source:** UK Civil Aviation Authority (CAA)

**Target Airports:** Heathrow, Gatwick, Manchester, Birmingham, Edinburgh

#### Objective
This notebook serves as the analytical foundation for the DSR dashboard artifact. The purpose of this EDA is to:
1. **Validate** the structural integrity of the automated SQLite extraction pipeline.
2. **Calculate** the core performance metrics (YoY Growth, Volatility, Market Share).
3. **Visualize** the post-pandemic recovery arcs to inform the UI/UX design of the final Streamlit application.

---
### Phase 1: Data Ingestion & Chronological Formatting
Before running aggregations, we must extract the master dataset from the `COMPLETE_aviation_dashboard.db` SQLite database. 

**Critical Transformation:** Standard string representations of months (e.g., "January", "February") will default to alphabetical sorting in Python visualization libraries. To preserve the time-series integrity, the `Month` column is converted into a strict, ordered `Categorical` data type.

In [16]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import sqlite3

sns.set_style("whitegrid")

In [ ]:
data = pd.read_csv("COMPLETE_aviation_dashboard.csv") # Extracted from csv for ease of access
data

,Year,Month,Airport,Total Passengers,International Passengers,Domestic Passengers
0,2021,1,BIRMINGHAM,33967,28569,5346
1,2021,1,EDINBURGH,43413,25581,17832
2,2021,1,GATWICK,88358,75800,12424
3,2021,1,HEATHROW,677356,617621,59735
4,2021,1,MANCHESTER,126963,108725,18196
...,...,...,...,...,...,...
235,2024,12,BIRMINGHAM,885913,789017,96896
236,2024,12,EDINBURGH,1153317,733962,419019
237,2024,12,GATWICK,3179796,2934372,245274
238,2024,12,HEATHROW,7077966,6650971,423997


In [18]:
data["Month"] = pd.to_datetime(data["Month"], format='%m').dt.strftime('%B')

In [19]:
data

,Year,Month,Airport,Total Passengers,International Passengers,Domestic Passengers
0,2021,January,BIRMINGHAM,33967,28569,5346
1,2021,January,EDINBURGH,43413,25581,17832
2,2021,January,GATWICK,88358,75800,12424
3,2021,January,HEATHROW,677356,617621,59735
4,2021,January,MANCHESTER,126963,108725,18196
...,...,...,...,...,...,...
235,2024,December,BIRMINGHAM,885913,789017,96896
236,2024,December,EDINBURGH,1153317,733962,419019
237,2024,December,GATWICK,3179796,2934372,245274
238,2024,December,HEATHROW,7077966,6650971,423997


In [20]:
data["Airport"].unique()

array(['BIRMINGHAM', 'EDINBURGH', 'GATWICK', 'HEATHROW', 'MANCHESTER'],
      dtype=object)

In [21]:
data[data.duplicated]

,Year,Month,Airport,Total Passengers,International Passengers,Domestic Passengers


In [22]:
months_order = ['January', 'February', 'March', 'April', 'May', 'June', 
                'July', 'August', 'September', 'October', 'November', 'December']

# Converting the Month column to a Categorical data type with a strict order
data['Month'] = pd.Categorical(data['Month'], categories=months_order, ordered=True)

---
### Phase 2: Pipeline Validation & Integrity Checks
In accordance with DSR evaluation principles, we must empirically verify the accuracy of the underlying dataset before extracting insights. 

We conduct two primary audits:
1. **The Equation Check:** Validating that `Domestic Passengers + International Passengers ≈ Total Passengers` (allowing a <5% variance for unrecorded transit passengers).
2. **The Zero Audit:** Identifying months where airports reported exactly `0` international passengers to highlight the severity of 2021 pandemic lockdowns.

In [23]:
print("--- 1. THE EQUATION CHECK ---")
# Check if Domestic + International roughly equals Total (allowing a 5% margin for transit passengers)
data['Calculated_Total'] = data['Domestic Passengers'] + data['International Passengers']
data['Discrepancy'] = abs(data['Total Passengers'] - data['Calculated_Total']) / data['Total Passengers']

# Find rows where the discrepancy is greater than 5%
anomalies = data[data['Discrepancy'] > 0.05]
print(f"Found {len(anomalies)} rows with >5% discrepancy between passenger splits and totals.")
if not anomalies.empty:
    print(anomalies[['Year', 'Month', 'Airport', 'Discrepancy']].head())

print("\n--- 2. THE ZERO AUDIT ---")
# Did any airport completely shut down international travel for a month?
zero_intl = data[data['International Passengers'] == 0]
print(f"Found {len(zero_intl)} months where an airport reported 0 international passengers.")
if not zero_intl.empty:
    print(zero_intl[['Year', 'Month', 'Airport']])

--- 1. THE EQUATION CHECK ---
Found 0 rows with >5% discrepancy between passenger splits and totals.

--- 2. THE ZERO AUDIT ---
Found 0 months where an airport reported 0 international passengers.


---
### Phase 3: Core Metrics & Statistical Analysis
This section generates the definitive statistics that measure the speed and stability of the aviation recovery. 

**Key Metrics Calculated:**
* **Year-over-Year (YoY) Growth:** Quantifying the "Revenge Travel" spike of 2022.
* **Market Share & Fleet Structure:** Analyzing the shift from domestic-heavy pandemic travel to international baseline normalization.
* **The Coefficient of Variation (CV):** Measuring system stability. *Note: A lower CV score indicates that the industry has recovered its predictable, seasonal rhythm relative to its total passenger volume.*

In [24]:
annual_totals = data.groupby('Year')['Total Passengers'].sum()

# Calculate the percentage change from the previous year
yoy_growth = annual_totals.pct_change() * 100
print(pd.DataFrame({'Total Passengers': annual_totals, 'YoY Growth (%)': yoy_growth.round(2)}))

      Total Passengers  YoY Growth (%)
Year                                  
2021          37248193             NaN
2022         138659363          272.26
2023         174053032           25.53
2024         186549032            7.18


In [25]:
split_df = data.groupby('Year')[['Domestic Passengers', 'International Passengers']].sum()
split_df['% Domestic'] = (split_df['Domestic Passengers'] / annual_totals * 100).round(2)
split_df['% International'] = (split_df['International Passengers'] / annual_totals * 100).round(2)
split_df[['% Domestic', '% International']]

,% Domestic,% International
Year,,
2021,15.74,84.22
2022,8.52,91.45
2023,8.41,91.56
2024,8.43,91.53


In [26]:
df_2024 = data[data['Year'] == 2024]
market_share = df_2024.groupby('Airport')['Total Passengers'].sum()
market_share_pct = (market_share / market_share.sum() * 100).round(2).sort_values(ascending=False)
print(market_share_pct)

Airport
HEATHROW      44.97
GATWICK       23.18
MANCHESTER    16.50
EDINBURGH      8.46
BIRMINGHAM     6.89
Name: Total Passengers, dtype: float64


---
### Phase 4: Visualizing the Recovery Trajectory
To inform the eventual Streamlit dashboard design, we prototype the two primary visual narratives of the dataset.

1. **The Macro View (The Recovery Arc):** A continuous time-series plotting the absolute growth from the depressed baseline of early 2021 to the normalized peaks of 2024.
2. **The Micro View (The Seasonality Return):** An overlapping 12-month timeline comparing each year's distinct shape.

In [ ]:
# Set the visual style
sns.set_theme(style="whitegrid")

# Insight 1: The "Revenge Travel" Recovery Curve
plt.figure(figsize=(12, 6))
# Group by Year and Month to get the total UK traffic per month
monthly_trend = data.groupby(['Year', 'Month'])['Total Passengers'].sum().reset_index()
# Create a continuous timeline column for plotting
monthly_trend['Timeline'] = monthly_trend['Month'].astype(str) + " " + monthly_trend['Year'].astype(str)

sns.lineplot(data=monthly_trend, x='Timeline', y='Total Passengers', marker='o')
plt.title('The Post-Pandemic Recovery Arc (2021 - 2024)', fontsize=14)
plt.xticks(rotation=90)
plt.ylabel('Total Passengers')
plt.tight_layout()
plt.show()

# Insight 2: The Return of Seasonality (The Bell Curve)
plt.figure(figsize=(12, 6))
# Plot each year as its own line on a single 12-month axis
sns.lineplot(data=monthly_trend, x='Month', y='Total Passengers', hue='Year', palette='tab10', marker='o')
plt.title('The Return of the Summer Peak (Seasonality Restored)', fontsize=14)
plt.ylabel('Total Passengers')
plt.tight_layout()
plt.show()

In [28]:

# Group by Airport and Year
airport_yearly = data.groupby(['Airport', 'Year'])['Total Passengers'].sum().unstack()

# Calculate the total growth from 2021 to 2024 for each airport
airport_yearly['Total Recovery Growth (%)'] = ((airport_yearly[2024] - airport_yearly[2021]) / airport_yearly[2021] * 100).round(2)

# Sort to see who grew the fastest
print(airport_yearly[['Total Recovery Growth (%)']].sort_values(by='Total Recovery Growth (%)', ascending=False))

Year        Total Recovery Growth (%)
Airport                              
GATWICK                        590.68
EDINBURGH                      421.67
BIRMINGHAM                     417.57
MANCHESTER                     405.97
HEATHROW                       332.52


In [29]:


# Calculate the mean and standard deviation for the 12 months of each year
stability = data.groupby('Year')['Total Passengers'].agg(['mean', 'std'])

# Coefficient of Variation (CV) = Standard Deviation / Mean
# A lower CV means the data is more stable and predictable
stability['Volatility Score (CV)'] = (stability['std'] / stability['mean']).round(3)
print(stability[['Volatility Score (CV)']])

      Volatility Score (CV)
Year                       
2021                  1.239
2022                  0.766
2023                  0.735
2024                  0.715


---
### Executive Summary & Next Steps
**Key Findings:**
> * **Gatwick** experienced the most aggressive relative growth, due to its near-total operational shutdown in 2021 compared to Heathrow's maintained baseline.
> * The UK Aviation market structure permanently normalized by **2022**, locking into an ~8.5% Domestic / 91.5% International split.
> * The **Volatility Score (CV)** successfully proves that by 2023/2024, the industry didn't just grow in volume, but recovered its systemic predictability (dropping to a stable CV of ~0.715).

**Next Steps (Dashboard Development):**
The validated data and metrics from this notebook will now be deployed into a dynamic Streamlit web application. The dashboard will feature strict constraint filters (Max 3 Airports), cached database querying for sub-3-second load times, and interactive Plotly visualizations.

In [ ]:
from pathlib import Path
desktop_path = Path(r"C:\Users\Kamiye\Desktop")
csv_path = desktop_path / "aviation_dashboard.csv"
db_path = desktop_path / "aviation_dashboard.db"

# Export to CSV
data.to_csv(csv_path, index=False)
print(f"SUCCESS: Master CSV saved to {csv_path}")
        
        
try:
    conn = sqlite3.connect(db_path)
    data.to_sql('passenger_trends', conn, if_exists='replace', index=False)
    conn.close()
    print(f"SUCCESS: Master Database securely written to {db_path}")
except Exception as e:
    print(f"Database Error: {e}")